# Self-Hosted Conversational SDR Voice Agent (Colab Prototype)

Fully self-hosted STT (Whisper) + LLM (Qwen2.5 via Ollama) + TTS (OmniVoice)
voice pipeline on LiveKit Agents, exposed via a self-hosted `livekit-server`
tunneled through ngrok. Test by connecting through LiveKit's Agents Playground.

Non-goals for this notebook: telephony, CRM, custom UI, compliance handling.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU in Colab"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
%pip install -q \
    livekit-agents~=1.0 \
    "livekit-plugins-openai~=1.0" \
    "livekit-plugins-silero~=1.0" \
    faster-whisper \
    ollama \
    pyngrok \
    soundfile

# OmniVoice: try PyPI first, fall back to installing from GitHub source
%pip install -q omnivoice || %pip install -q "git+https://github.com/k2-fsa/OmniVoice.git"

In [ ]:
import livekit.agents
import livekit.plugins.openai
import livekit.plugins.silero
import faster_whisper
import omnivoice
import ollama
import pyngrok
print("All imports OK")

In [ ]:
!curl -sSL https://get.livekit.io | bash
!livekit-server --version

In [ ]:
import urllib.request

REFERENCE_AUDIO_PATH = "jfk.flac"
REFERENCE_TRANSCRIPT_SNIPPET = "ask not what your country can do for you"

urllib.request.urlretrieve(
    "https://github.com/openai/whisper/raw/main/tests/jfk.flac",
    REFERENCE_AUDIO_PATH,
)
print("Downloaded:", REFERENCE_AUDIO_PATH)

In [ ]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
print("Whisper model loaded")

In [ ]:
segments, info = whisper_model.transcribe(REFERENCE_AUDIO_PATH, language="en")
transcript = " ".join(segment.text for segment in segments).strip()
print("Transcript:", transcript)
assert REFERENCE_TRANSCRIPT_SNIPPET in transcript.lower(), f"Expected snippet not found in: {transcript}"
print("STT verification PASSED")

In [ ]:
import subprocess
import time

!curl -fsSL https://ollama.com/install.sh | sh

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
print("Ollama server starting, PID:", ollama_process.pid)

In [ ]:
!ollama pull qwen2.5:7b-instruct

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "qwen2.5:7b-instruct",
        "messages": [{"role": "user", "content": "Reply with exactly the word: PONG"}],
        "max_tokens": 10,
    },
    timeout=60,
)
response.raise_for_status()
reply = response.json()["choices"][0]["message"]["content"]
print("LLM reply:", reply)
assert "PONG" in reply.upper(), f"Unexpected reply: {reply}"
print("LLM verification PASSED")

In [ ]:
import torch
from omnivoice import OmniVoice

omnivoice_model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)
print("OmniVoice model loaded")

In [ ]:
import numpy as np
import soundfile as sf

TEST_SENTENCE = "Hi, this is a quick test of the voice pipeline."

audio_chunks = omnivoice_model.generate(
    text=TEST_SENTENCE,
    instruct="female, medium pitch, american accent, friendly sales tone",
)
audio = audio_chunks[0]

assert isinstance(audio, np.ndarray), f"Expected np.ndarray, got {type(audio)}"
assert audio.ndim == 1 and audio.shape[0] > 0, f"Unexpected shape: {audio.shape}"

duration_seconds = audio.shape[0] / 24000
rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
print(f"Duration: {duration_seconds:.2f}s, RMS: {rms:.4f}")
assert duration_seconds > 0.5, "Audio too short — synthesis likely failed"
assert rms > 0.001, "Audio is near-silent — synthesis likely failed"

sf.write("tts_test_output.wav", audio, 24000)
print("TTS verification PASSED — listen to tts_test_output.wav to confirm quality")

In [ ]:
from IPython.display import Audio, display
display(Audio("tts_test_output.wav"))